# 🎬 CorridorKey 2.0 — Photo Background Remover

Upload one photo and get a **transparent RGBA PNG** of the foreground. Supported inputs: PNG, JPG, WebP, TIFF, BMP (anything Pillow can read).

**Quick start:** click **Runtime → Run all**, then choose your photo when the file picker appears. The defaults already produce the transparent result; you do not need to change any setting.

**What you need:** a Colab GPU runtime (**Runtime → Change runtime type → Hardware accelerator: GPU**). A free T4 can run via a local float16 compatibility patch (experimental and slow); an A100/L4 is recommended. The first run installs PyTorch and downloads ~5 GB of model weights (about 10–15 minutes). Later runs in the same session skip that.

**What happens:** the notebook clones [CorridorKey 2.0](https://github.com/nikopueringer/CorridorKey2.0), runs the official `infer.py` on a one-frame directory, and extracts the model's `cutout` output as a PNG with alpha. The subject is chosen automatically unless you provide a hint in the optional section.

<details>
<summary><b>License (short version)</b></summary>

CorridorKey 2.0 was designed and trained by Zachary Harris in collaboration with Niko and Corridor Digital. You may use it and the outputs for personal or commercial work. You may not sell the tool, its code, or its model, and you may not offer it as a paid API/inference service without permission. If you build on it, your release must stay free, keep the same license, and keep the CorridorKey name. Organizations with more than US $20M annual revenue need a plugin seat per user. Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Donationware and pay-what-you-want are fine.

</details>


In [ ]:
#@title 1. Check GPU { display-mode: "form" }
import subprocess

smi = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True,
)
print(smi.stdout.strip() or "nvidia-smi did not report a GPU.")

try:
    import torch
except Exception as exc:
    raise SystemExit(f"PyTorch could not be imported: {exc}")

if not torch.cuda.is_available():
    raise SystemExit(
        "No CUDA GPU is available.\n"
        "In Colab: Runtime > Change runtime type > Hardware accelerator > GPU, then re-run this cell."
    )

props = torch.cuda.get_device_properties(0)
vram_gb = props.total_memory / 1e9
print(f"GPU: {props.name} | VRAM: {vram_gb:.1f} GB | compute capability: {props.major}.{props.minor}")
print(f"PyTorch (before install): {torch.__version__}")
if props.major < 7:
    raise SystemExit(
        "This GPU is too old for the CUDA 12.8 PyTorch build used here "
        "(minimum compute capability 7.0). Use a T4, A100, L4, or newer."
    )
if props.major < 8:
    print(
        "\nNOTE: pre-Ampere GPU (T4/Turing) detected. T4 has no native bfloat16, so the setup cell "
        "applies a local float16 compatibility patch. This is experimental; an A100/L4 is recommended "
        "for production and large jobs."
    )
if vram_gb < 7:
    print("\nWARNING: less than 7 GB of VRAM. Low-VRAM mode will be used automatically; very large photos may not fit.")


In [ ]:
#@title 2. Upload your photo { display-mode: "form" }
import gc
import re
from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image, ImageOps

uploaded = files.upload()
if not uploaded:
    raise SystemExit("No file was selected. Re-run this cell and choose a photo.")
if len(uploaded) > 1:
    print("You selected more than one file; this notebook processes one photo per run. Using the first.")

raw_name = next(iter(uploaded))
raw_path = Path(raw_name)
if not raw_path.is_file():
    raw_path = Path("/content") / raw_name
    raw_path.write_bytes(uploaded[raw_name])

stem = Path(raw_name).stem
safe = re.sub(r"[^A-Za-z0-9._-]+", "_", stem).strip("._") or "photo"
PHOTO_NAME = safe
work = Path("/content/work") / PHOTO_NAME
work.mkdir(parents=True, exist_ok=True)

img = Image.open(raw_path)
img = ImageOps.exif_transpose(img)
try:
    if img.info.get("icc_profile"):
        import io
        from PIL import ImageCms
        srgb = ImageCms.createProfile("sRGB")
        src_profile = ImageCms.ImageCmsProfile(io.BytesIO(img.info["icc_profile"]))
        img = ImageCms.profileToProfile(img, src_profile, srgb, outputMode="RGB")
except Exception as exc:
    print("ICC profile conversion skipped:", exc)
img = img.convert("RGB")

PHOTO_SOURCE = work / "original_srgb.png"
img.save(PHOTO_SOURCE)
print("Saved:", PHOTO_SOURCE)
print("Resolution:", img.width, "x", img.height)
thumb = img.copy()
thumb.thumbnail((1024, 1024), Image.LANCZOS)
display(thumb)

del uploaded
gc.collect()


In [ ]:
#@title 3. Install CorridorKey 2.0 (first run ~10–15 min) { display-mode: "form" }
#@markdown Clones the repo, installs the tested CUDA PyTorch + dependencies, checks ffmpeg, and downloads the ~5 GB weights.
#@markdown On T4/Turing GPUs it also applies a local float16 compatibility patch (experimental), because those GPUs lack native bfloat16.
#@markdown Re-running is safe and skips finished steps.
force_reinstall = False #@param {type:"boolean"}

import json
import shutil
import subprocess
import sys
from pathlib import Path

REPO_DIR = Path("/content/CorridorKey2.0")
REPO_URL = "https://github.com/nikopueringer/CorridorKey2.0.git"
MARKER = Path("/content/.corridorkey_setup_v1.json")
TORCH_VERSION = "2.9.1"
TV_VERSION = "0.24.1"
PATCH_FILES = ("corridorkey/runtime.py", "corridorkey/video_io.py")
PATCH_MARKER = REPO_DIR / ".t4_fp16_patch"


def run(cmd, cwd=None):
    print("$", " ".join(str(c) for c in cmd), flush=True)
    subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None, check=True)


def torch_state():
    code = (
        "import torch, torchvision; "
        "print(torch.__version__, torchvision.__version__, int(torch.cuda.is_available()))"
    )
    try:
        out = subprocess.check_output([sys.executable, "-c", code], text=True).strip().split()
        if len(out) >= 3:
            return out[0], out[1], out[2] == "1"
    except Exception:
        pass
    return None, None, False


# Restore upstream files if a previous T4 patch is present, so git can update cleanly.
if (REPO_DIR / "infer.py").is_file() and PATCH_MARKER.is_file():
    subprocess.run(
        ["git", "-C", str(REPO_DIR), "checkout", "--", *PATCH_FILES],
        capture_output=True, text=True,
    )
    try:
        PATCH_MARKER.unlink()
    except FileNotFoundError:
        pass
    print("Restored upstream files before updating.", flush=True)

# 1) Repository
if not (REPO_DIR / "infer.py").is_file():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)])
else:
    try:
        run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"])
    except subprocess.CalledProcessError:
        print("Could not update the existing checkout; continuing with it.", flush=True)

commit = subprocess.check_output(
    ["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True
).strip()
print("CorridorKey commit:", commit, flush=True)

marker = {}
if MARKER.is_file():
    try:
        marker = json.loads(MARKER.read_text())
    except Exception:
        marker = {}
same_setup = (not force_reinstall and marker.get("commit") == commit)

# 2) PyTorch
torch_v, tv_v, cuda_ok = torch_state()
need_torch = (
    force_reinstall
    or torch_v is None
    or not torch_v.startswith(TORCH_VERSION)
    or tv_v is None
    or not tv_v.startswith(TV_VERSION)
    or not cuda_ok
)
if need_torch:
    print(f"Installing PyTorch {TORCH_VERSION} + torchvision {TV_VERSION} (CUDA 12.8)...", flush=True)
    run([
        sys.executable, "-m", "pip", "install", "-q",
        f"torch=={TORCH_VERSION}", f"torchvision=={TV_VERSION}",
        "--index-url", "https://download.pytorch.org/whl/cu128",
    ])
else:
    print(f"PyTorch {torch_v} + torchvision {tv_v} already installed; skipping.", flush=True)

# 2b) T4/Turing compatibility: bfloat16 is not native on pre-Ampere GPUs.
try:
    cap_major = int(subprocess.check_output(
        [sys.executable, "-c", "import torch; print(torch.cuda.get_device_capability(0)[0])"],
        text=True,
    ).strip())
except Exception:
    cap_major = 8

if cap_major < 8:
    patched = []
    for rel in PATCH_FILES:
        p = REPO_DIR / rel
        text = p.read_text(encoding="utf-8")
        if "torch.bfloat16" in text:
            p.write_text(text.replace("torch.bfloat16", "torch.float16"), encoding="utf-8")
            patched.append(rel)
    PATCH_MARKER.write_text("fp16\n", encoding="utf-8")
    print("T4/Turing detected: applied the local float16 compatibility patch.", flush=True)
    print("Patched:", ", ".join(patched) if patched else "already patched", flush=True)
else:
    print("Ampere+ GPU detected: using the native bfloat16 path.", flush=True)

# 3) Python dependencies
deps_ok = subprocess.run(
    [sys.executable, "-c", "import cv2, safetensors, transformers, timm, einops, scipy, utils3d"],
    capture_output=True,
).returncode == 0
if force_reinstall or not same_setup or not deps_ok:
    print("Installing Python dependencies...", flush=True)
    run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")])
else:
    print("Python dependencies already installed for this commit; skipping.", flush=True)

# 4) ffmpeg + ProRes encoder
if shutil.which("ffmpeg") is None:
    run(["apt-get", "-qq", "install", "-y", "ffmpeg"])
encoders = subprocess.check_output(
    ["ffmpeg", "-hide_banner", "-encoders"], text=True, errors="ignore"
)
if "prores_ks" not in encoders:
    raise RuntimeError(
        "This ffmpeg build has no prores_ks encoder. The transparent .mov output cannot be written. "
        "Re-run: apt-get install -y ffmpeg"
    )
print("ffmpeg + prores_ks OK", flush=True)

# 5) Weights
required = [
    REPO_DIR / "weights/corridorkey_v2.safetensors",
    REPO_DIR / "weights/foundation_stats.safetensors",
    REPO_DIR / "vfms/moge/model.pt",
]
cradio_dir = REPO_DIR / "vfms/cradio/C-RADIOv4-SO400M"
rvm_dir = REPO_DIR / "vfms/rvm"
missing = [p for p in required if not p.is_file()]
if not cradio_dir.is_dir() or not any(cradio_dir.iterdir()):
    missing.append(cradio_dir)
if not rvm_dir.is_dir() or not any(rvm_dir.glob("*.npz")):
    missing.append(rvm_dir)

if force_reinstall or missing:
    print("Downloading model weights (~5 GB, resumable)...", flush=True)
    run([sys.executable, str(REPO_DIR / "scripts/download_weights.py")], cwd=REPO_DIR)
else:
    print("Model weights already present; skipping.", flush=True)

# 6) Done
MARKER.write_text(json.dumps({
    "commit": commit,
    "torch": TORCH_VERSION,
    "torchvision": TV_VERSION,
}))
print("\nSetup complete.", flush=True)
print("Repo:", REPO_DIR, flush=True)
print(
    "If Colab shows a 'restart runtime' banner, you can ignore it: "
    "inference runs in a fresh subprocess.",
    flush=True,
)


In [ ]:
#@title ⚙️ Advanced settings (optional) { display-mode: "form" }
#@markdown The defaults produce one clean transparent PNG. Open this only if you need more.
save_alpha = False #@param {type:"boolean"}
#@markdown Also save the black/white matte.
save_fg = False #@param {type:"boolean"}
#@markdown Also save the raw foreground RGB (no alpha).
save_checker = False #@param {type:"boolean"}
#@markdown Also save a full-resolution foreground-over-checker render.
hint_quality = 0.95 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Trust in the optional matte: 0.95 = clean artist matte, 0.75 = rough, <=0.5 = loose. Ignored without a matte.
despill_strength = 0.5 #@param {type:"slider", min:0.0, max:1.0, step:0.05}
#@markdown Green/blue spill removal in the cutout. 0 = none, 1 = full.
max_long_side = 0 #@param {type:"integer"}
#@markdown Resize the photo so its longest side is at most this, before inference. 0 = keep original; e.g. 4096 for huge photos.
cutout_linear = False #@param {type:"boolean"}
#@markdown Export straight scene-linear RGB instead of sRGB. For linear VFX compositing only; normal viewers will look wrong.


In [ ]:
#@title 4. Optional: upload a first-frame matte { display-mode: "form" }
#@markdown A matte is a black/white image: **white = subject**, **black = background**. If the image has an alpha channel, the alpha is used.
#@markdown For a single photo this is effectively a full hint. If you don't have one, leave this off.
ADD_HINT = False #@param {type:"boolean"}

from pathlib import Path

from google.colab import files
from IPython.display import display
from PIL import Image

if "PHOTO_NAME" not in globals():
    raise SystemExit("Run the photo upload cell first.")

HINT_SOURCE = None
if ADD_HINT:
    uploaded_hint = files.upload()
    if not uploaded_hint:
        print("No hint file selected; continuing without a hint.")
    else:
        hint_name = next(iter(uploaded_hint))
        hint_path = Path(hint_name)
        if not hint_path.is_file():
            hint_path = Path("/content") / hint_name
            hint_path.write_bytes(uploaded_hint[hint_name])
        HINT_SOURCE = Path("/content/work") / PHOTO_NAME / "hint_source.png"
        try:
            hint_img = Image.open(hint_path)
        except Exception as exc:
            raise SystemExit(f"Could not read the hint image: {exc}")
        hint_img.save(HINT_SOURCE)
        print(f"Hint saved: {HINT_SOURCE} ({hint_img.width} x {hint_img.height}, mode {hint_img.mode})")
        thumb = hint_img.copy()
        thumb.thumbnail((1024, 1024), Image.LANCZOS)
        display(thumb)
else:
    print("No hint will be used. The model will choose the foreground subject.")


In [ ]:
#@title 5. Remove the background { display-mode: "form" }
import shutil
import subprocess
import sys
import time
from pathlib import Path

from PIL import Image

if "PHOTO_SOURCE" not in globals() or not Path(PHOTO_SOURCE).is_file():
    raise SystemExit("Run the photo upload cell first.")

save_alpha = bool(globals().get("save_alpha", False))
save_fg = bool(globals().get("save_fg", False))
save_checker = bool(globals().get("save_checker", False))
hint_quality = float(globals().get("hint_quality", 0.95))
despill_strength = float(globals().get("despill_strength", 0.5))
max_long_side = max(0, int(globals().get("max_long_side", 0)))
cutout_linear = bool(globals().get("cutout_linear", False))

REPO_DIR = Path(globals().get("REPO_DIR", "/content/CorridorKey2.0"))
infer_py = REPO_DIR / "infer.py"
if not infer_py.is_file():
    raise SystemExit("CorridorKey is not installed. Run the setup cell above.")

work = Path("/content/work") / PHOTO_NAME
input_dir = work / "input"
input_dir.mkdir(parents=True, exist_ok=True)
input_path = input_dir / "0001.png"

img = Image.open(PHOTO_SOURCE).convert("RGB")
original_size = img.size
if max_long_side and max(img.size) > max_long_side:
    scale = max_long_side / max(img.size)
    new_size = (max(1, round(img.width * scale)), max(1, round(img.height * scale)))
    img = img.resize(new_size, Image.LANCZOS)
    print(f"Resized input from {original_size} to {new_size}.")
img.save(input_path)

hint_path = None
hint_src = globals().get("HINT_SOURCE")
if hint_src and Path(hint_src).is_file():
    hint = Image.open(hint_src)
    hint = hint.convert("RGBA") if hint.mode in ("RGBA", "LA") else hint.convert("L")
    if hint.size != img.size:
        hint = hint.resize(img.size, Image.LANCZOS)
        print(f"Resized hint to the input size ({img.width} x {img.height}).")
    hint_path = work / "hint.png"
    hint.save(hint_path)
    print("Using hint:", hint_path)

outputs = ["cutout"]
if save_alpha:
    outputs.append("alpha")
if save_fg:
    outputs.append("fg")
if save_checker:
    outputs.append("checker")

out_dir = Path("/content/outputs") / PHOTO_NAME
if out_dir.exists():
    shutil.rmtree(out_dir)
out_dir.mkdir(parents=True, exist_ok=True)

cmd = [
    sys.executable, str(infer_py),
    "--input", str(input_dir),
    "--output_dir", str(out_dir),
    "--outputs", *outputs,
    "--frame_dir_fps", "24",
    "--hint_quality", str(hint_quality),
    "--despill_strength", str(despill_strength),
]
if hint_path is not None:
    cmd += ["--hint_first_frame", str(hint_path)]
if cutout_linear:
    cmd.append("--cutout_linear")

print("\nCommand:")
print(" ".join(cmd))
print("Output directory:", out_dir)
print()
start = time.time()
try:
    subprocess.run(cmd, check=True)
except subprocess.CalledProcessError as exc:
    print("\nInference failed with exit code", exc.returncode)
    print("Common fixes:")
    print("- CUDA out of memory: lower max_long_side (e.g. 4096 or 2048), then re-run.")
    print("- Missing weights: re-run the setup cell.")
    print("- No GPU: Runtime > Change runtime type > GPU.")
    raise
print(f"\nDone in {time.time() - start:.1f} s.")
print("Outputs are in:", out_dir)

OUT_DIR = out_dir
CLIP_NAME = PHOTO_NAME


In [ ]:
#@title 6. Preview & download { display-mode: "form" }
import subprocess
import zipfile
from pathlib import Path

import numpy as np
from IPython.display import FileLink, Image as IPImage, display
from PIL import Image

if "OUT_DIR" not in globals():
    raise SystemExit("Run the inference cell first.")

out_dir = Path(OUT_DIR)
clip = CLIP_NAME


def human(n):
    n = float(n)
    for unit in ("B", "KB", "MB", "GB"):
        if n < 1024 or unit == "GB":
            return f"{n:.1f} {unit}"
        n /= 1024


def checker_rgba(width, height, tile=32):
    yy, xx = np.indices((height, width))
    mask = (((xx // tile) + (yy // tile)) % 2).astype(np.uint8)
    gray = np.where(mask == 1, 210, 160).astype(np.uint8)
    rgb = np.repeat(gray[:, :, None], 3, axis=2)
    return Image.fromarray(rgb, "RGB").convert("RGBA")


def show_image(path, max_side=1024):
    img = Image.open(path)
    img.thumbnail((max_side, max_side), Image.LANCZOS)
    display(img)


def extract_first_frame(video_path, png_path):
    subprocess.run(
        ["ffmpeg", "-y", "-loglevel", "error", "-i", str(video_path),
         "-frames:v", "1", "-pix_fmt", "rgba", str(png_path)],
        check=True,
    )


items = []
cutout_mov = out_dir / f"{clip}_cutout.mov"
cutout_png = out_dir / f"{clip}_cutout.png"

if cutout_mov.is_file():
    extract_first_frame(cutout_mov, cutout_png)
    rgba = Image.open(cutout_png).convert("RGBA")
    bg = checker_rgba(rgba.width, rgba.height)
    preview = Image.alpha_composite(bg, rgba).convert("RGB")
    preview_path = out_dir / f"{clip}_checker_preview.png"
    preview.save(preview_path)
    print("Transparent PNG:", cutout_png, f"({human(cutout_png.stat().st_size)})")
    print("Resolution:", rgba.width, "x", rgba.height)
    print("Alpha range:", rgba.getchannel("A").getextrema())
    if globals().get("cutout_linear", False):
        print("NOTE: cutout_linear was enabled; PNG values are scene-linear, not sRGB.")
    print("\nResult over a checkerboard:")
    show_image(preview_path, 1024)
    items.extend([cutout_png, preview_path])
else:
    print("No cutout output found. Check the inference log above.")

for name, filename in (("alpha", f"{clip}_alpha.mp4"), ("fg", f"{clip}_fg.mp4")):
    video = out_dir / filename
    if video.is_file():
        png = out_dir / f"{clip}_{name}.png"
        extract_first_frame(video, png)
        print(f"{name}: {png} ({human(png.stat().st_size)})")
        show_image(png, 512)
        items.append(png)

for video in sorted(out_dir.glob(f"{clip}_checker_comp_despill*.mp4")):
    png = out_dir / f"{clip}_checker.png"
    extract_first_frame(video, png)
    print("checker:", png, f"({human(png.stat().st_size)})")
    show_image(png, 512)
    items.append(png)

original = Path(PHOTO_SOURCE)
if original.is_file():
    items.append(original)

unique = []
seen = set()
for p in items:
    p = Path(p)
    if p.is_file() and str(p) not in seen:
        unique.append(p)
        seen.add(str(p))

zip_path = out_dir / f"{clip}_result.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in unique:
        zf.write(p, p.name)
print("\nZIP:", zip_path, f"({human(zip_path.stat().st_size)})")

if cutout_png.is_file():
    display(FileLink(str(cutout_png), result_html_prefix="Download transparent PNG: "))
display(FileLink(str(zip_path), result_html_prefix="Download ZIP: "))


<details>
<summary><b>⚙️ Parameter reference — what each advanced setting does</b></summary>

The transparent `cutout` is always written. Everything below is optional.

| Setting | Default | Effect |
|---|---|---|
| `save_alpha` | off | Also save the black/white matte as `..._alpha.png`. |
| `save_fg` | off | Also save the raw foreground RGB as `..._fg.png` (no alpha). |
| `save_checker` | off | Also save the model's full-resolution foreground-over-checker render. |
| `hint_quality` | 0.95 | How much to trust an uploaded matte. 0.95 = clean artist matte; 0.75 = rough automated mask; ≤0.5 = loose/scribble. Ignored when no matte is uploaded. |
| `despill_strength` | 0.5 | Removes green/blue spill from the cutout. 0 = none, 1 = full. |
| `max_long_side` | 0 | Resize the input before inference so its longest side is at most this many pixels. 0 keeps the original. Use 4096 or 2048 for huge photos. The output matches the resized input. |
| `cutout_linear` | off | Writes straight scene-linear RGB instead of sRGB. For linear VFX compositing only; normal image viewers will look wrong. |

**Hints.** A matte is a black/white image: white means subject, black means background. If the image has an alpha channel, the alpha is used. The notebook resizes the matte to the photo automatically. For one photo, a first-frame hint is effectively a full hint.

**What the notebook runs.** The hidden command is equivalent to:

```bash
python infer.py \
  --input /content/work/<name>/input \
  --output_dir /content/outputs/<name> \
  --outputs cutout [alpha] [fg] [checker] \
  --frame_dir_fps 24 \
  --hint_quality 0.95 \
  --despill_strength 0.5 \
  [--hint_first_frame /content/work/<name>/hint.png] \
  [--cutout_linear]
```

Native resolution is preserved. The model pads internally and crops back to the original size.

</details>


<details>
<summary><b>🛠️ Troubleshooting</b></summary>

**No GPU / “requires a CUDA GPU”.** In Colab: *Runtime → Change runtime type → Hardware accelerator → GPU*, then re-run the check cell.

**Colab asks to restart after installing PyTorch.** You can ignore it: inference runs in a fresh subprocess with the new PyTorch. If you do restart, re-run from the GPU check; the setup cell skips finished work.

**CUDA out of memory.** Lower `max_long_side` to 4096 or 2048, then re-run. For very large images, 2048 is a safe choice. A high-RAM Colab runtime also helps.

**T4 / bfloat16.** Pre-Ampere GPUs (T4/Turing) lack native bfloat16. The setup cell automatically patches the two runtime files to use float16 and prints a notice. This is experimental but uses the same 2-byte precision; if you hit a CUDA error, switch to an A100/L4 runtime.

**The run seems slow.** The first run loads the model and downloads weights. A T4 is slow; A100/L4 is much faster. The photo path only processes one frame, so the model load dominates.

**The session disconnected.** `/content` is erased when the session ends. Download the PNG/ZIP before disconnecting. The notebooks intentionally do not use Google Drive.

**The subject is wrong.** Upload a first-frame matte in the optional cell: white = subject, black = background, then re-run inference. Or use the CLI locally with a full hint.

**Edges have green/blue spill.** Raise `despill_strength` toward 1.0. For a stronger result, provide a cleaner hint.

**My photo is rotated.** The notebook applies EXIF orientation automatically. If it still looks wrong, rotate the source image before uploading.

**I need HDR / scene-linear output.** This notebook extracts an 8-bit sRGB PNG from the model's ProRes 4444 cutout. For unclipped linear EXR, use `infer.py --output_format exr` from the CLI; see the repo README.

**My image has a Log / ACES / HDR profile.** The runtime expects sRGB/Rec.709-like display-coded input. Convert upstream to sRGB before uploading.

</details>


<details>
<summary><b>📜 License & acknowledgements</b></summary>

CorridorKey 2.0 was designed and trained by **Zachary Harris**, in direct collaboration with **Niko** and **Corridor Digital**.

- You may use CorridorKey and its outputs for personal or commercial work. What you make is yours.
- You may not sell the tool, its code, or its model, or offer it as a paid API/inference service, without permission from Corridor Digital.
- If you build on the code or model, your release must be freely accessible, stay under the same license, and keep the CorridorKey name. Donationware/pay-what-you-want is fine.
- Organizations with more than US $20 million in annual revenue need to purchase a CorridorKey plugin seat for each person using the tool.
- Commercial software integrations: contact contact@corridordigital.com.

Full terms: [LICENSE](https://github.com/nikopueringer/CorridorKey2.0/blob/master/LICENSE). Official plugin/app: [corridorkey.com](https://corridorkey.com).

CorridorKey builds on three frozen foundation models:

- **C-RADIO** by NVIDIA — NVIDIA Open Model License
- **MoGe** by Microsoft — MIT
- **RVM** by Google DeepMind — code Apache 2.0, weights CC-BY 4.0

Details are in [THIRD_PARTY_NOTICES.md](https://github.com/nikopueringer/CorridorKey2.0/blob/master/THIRD_PARTY_NOTICES.md).

</details>
